# Credit-Risk Latent Factor Modeling with PROC HPPLS

## Executive Summary

A retail bank wants to predict three correlated credit-risk outcomes — card utilization, debt-to-income trajectory, and a default-probability proxy — from a wide set of highly collinear bureau-style and macroeconomic predictors. Ordinary regression breaks down under this collinearity, so this notebook uses **PROC HPPLS** (high-performance partial least squares) to extract a handful of latent factors that jointly explain the predictors and all three responses, then validates the factor count with a van der Voet cross-validation test on a held-out portfolio segment.

## Data Sources

All data is generated synthetically inside the notebook with `call streaminit(20240531)` — no external files or network access.

| Dataset | Rows | Variable | Role | Description |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unique customer key carried to the scored output |
| | | `Segment` | CLASS predictor | Portfolio segment: Retail, Affluent, Business |
| | | `b1`–`b12` | Predictors | 12 collinear monthly bureau-style behavioral signals |
| | | `RatePctChg` | Predictor | Customer-level interest-rate change exposure |
| | | `InquiryCount` | Predictor | Count of recent hard credit inquiries |
| | | `Utilization` | Response | Revolving credit utilization (%) |
| | | `DTIChange` | Response | Change in debt-to-income ratio |
| | | `DefaultProp` | Response | Default-probability proxy (0–1) |
| | | `Role` | Partition | TRAIN (~75%) / TEST (~25%) validation flag |

# Credit-Risk Latent Factor Modeling with PROC HPPLS

Banks routinely face the **wide, collinear predictor** problem: dozens of monthly bureau attributes, macroeconomic exposures, and behavioral signals that move together, used to predict several risk outcomes that are themselves correlated. Ordinary least squares is unstable here because the predictor matrix is near-singular.

**Partial least squares (PLS)** solves this by extracting a small number of latent factors from the *cross-covariance* of the predictors (X) and the responses (Y), so the factors are tuned to predict the outcomes — not just to summarize X. `PROC HPPLS` is the high-performance counterpart to `PROC PLS`, adding multithreaded execution, data partitioning for validation, and the van der Voet randomization test for choosing the number of factors.

In this notebook we build a single PLS model that simultaneously predicts three correlated credit-risk responses and use a held-out validation segment to confirm how many latent factors the data actually support.

## Step 1 — Generate a synthetic credit-risk panel

We simulate 600 customers. Two unobserved latent drivers (`stress` and `tenure`) generate twelve collinear bureau signals `b1`–`b12` plus rate and inquiry exposures — exactly the high-correlation structure PLS is designed for. The three responses (`Utilization`, `DTIChange`, `DefaultProp`) are different linear combinations of the same drivers, so they too are correlated. A `Role` flag holds out roughly a quarter of the book as a validation segment.

In [1]:
data credit;
   call streaminit(20240531);
   length Segment $8 Role $5;
   do CustomerID = 1 to 600;
      /* customer segment (categorical predictor) */
      u = rand('uniform');
      if u < 0.34 then Segment = 'Retail';
      else if u < 0.70 then Segment = 'Affluent';
      else Segment = 'Business';

      /* unobserved macro / behavioral drivers (collinear) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 collinear monthly bureau-style predictors b1-b12 */
      array b{12} b1-b12;
      do j = 1 to 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      end;

      /* macro covariates, also tied to the drivers */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( max(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* three correlated credit-risk responses */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      if DefaultProp < 0 then DefaultProp = 0;

      /* hold out ~25% as a validation segment */
      if rand('uniform') < 0.25 then Role = 'TEST';
      else Role = 'TRAIN';

      output;
   end;
   drop u stress tenure j;
run;

NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


## Step 2 — Fit the multi-response PLS model with cross-validation

The core call demonstrates the principal `PROC HPPLS` statements and the options a risk modeler would actually reach for:

- **`MODEL`** lists all three responses on the left and the full collinear predictor set on the right; **`/ solution`** prints the final predictive coefficients on the raw scale.
- **`CLASS Segment`** brings the portfolio segment in as a categorical predictor (it must precede `MODEL`).
- **`ID CustomerID`** carries the customer key into the scored output.
- **`CVTEST(stat=press ...)`** runs the van der Voet randomization test to pick the number of factors objectively rather than by eye; `seed=` makes it reproducible.
- **`PARTITION rolevar=Role(...)`** fits and scores on the training rows and holds out the `TEST` segment so the cross-validation PRESS is measured out-of-sample.
- **`VARSS`** and **`DETAILS`** report how much X- and Y-variation each successive factor explains.
- **`OUTPUT`** writes predicted values, residuals, X-scores, and PRESS for the fitted (training) rows to a scored dataset, keyed by `CustomerID`.

In [2]:
proc hppls data=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   class Segment;
   id CustomerID;
   model Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / solution;
   partition rolevar=Role(train='TRAIN' test='TEST');
   output out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
run;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class SEGMENT: 3 levels: Affluent Business Retail

Response Variable(s): UTILIZATION DTICHANGE DEFAULTPROP
Predictor Variable(s): B1 B2 B3 B4 B5 B6 B7 B8 B9 B10 B11 B12 RATEPCTCHG INQUIRYCOUNT SEGMENT

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8      0.6437 94.9768      0.1488 77.9714

Variation Accounted for by Variab

NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Step 3 — Summarize the predicted risk profile

With the model fit, we profile the predicted outcomes across the book. `PROC MEANS` reports the central tendency and spread of each predicted response so the risk team can sanity-check the scale (e.g., predicted utilization centered in the mid-40s, default proxy near the assumed 8% base rate).

In [3]:
proc means data=scored mean std min max maxdec=3;
   var Pred_Utilization Pred_DTIChange Pred_DefaultProp;
run;

                                                  The MEANS Procedure

 Variable                   Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------
 Pred_UTILIZATION         47.405      10.899      29.217      82.594
 Pred_DTICHANGE            0.649       2.503      -3.921       9.192
 Pred_DEFAULTPROP          0.090       0.049       0.008       0.235
 -------------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 4 — Inspect individual scored customers

Finally we list a few customers from the fitted **training** segment with their original `Role` flag, predicted utilization, and residual. (The held-out `TEST` rows are deliberately not scored, so we filter to `Role='TRAIN'` to show fully populated rows.) This is the kind of row-level output an analyst would attach to a model-monitoring report or feed downstream to a limit-setting engine.

In [4]:
proc print data=scored(obs=8);
   where Role = 'TRAIN';
   var CustomerID Role Pred_Utilization Resid_Utilization;
run;


  Obs  CustomerID   Role  PRED_UTILIZATION  RESID_UTILIZATION
-----  ----------  -----  ----------------  -----------------
    1           2  TRAIN     38.9211679221      11.0221020873
    2           3  TRAIN     40.0854583937       0.4204201474
    3           5  TRAIN      38.281005681      -6.6274996377
    4           6  TRAIN     62.3030569006      -1.1616371125
    5           7  TRAIN     42.3305742599      -2.1880063944
    6          10  TRAIN     52.1188381508       9.6850916555
    7          11  TRAIN     52.1656894765       2.6649623332
    8          12  TRAIN     51.2062446736       4.1920024228

... more observations (showing 8)



NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 8 observations printed, 4 variables


## Interpreting the results

The **Percent Variation Accounted for** table shows the first factor alone absorbs roughly three-quarters of the predictor variation (74.07%, the dominant collinear "stress" direction), while the second and third factors are where most of the *response* variation gets explained (37.94% and 10.46%, versus only 25.83% from the first) — the hallmark of PLS reorienting components toward prediction rather than pure X-variance. By eight factors the per-response R-squares settle at 0.81 (Utilization), 0.78 (DTIChange), and 0.74 (DefaultProp), confirming the three credit-risk outcomes are well captured by a low-dimensional latent structure.

The **van der Voet PRESS cross-validation** test is the decision-maker here: PRESS on the held-out segment drops sharply through the first four factors (8816 → 4772 → 3318 → 3244) and then flattens and ticks back up, so the test selects **four factors** as the parsimonious model. Extracting more would chase noise in the wide bureau block and degrade out-of-sample performance — precisely the overfitting a credit-risk model must avoid before deployment.

The `SOLUTION` coefficients give the bank an interpretable, regularized linear scorecard on the original variable scale, with `RatePctChg` (≈0.80, 0.88, 0.63 across the three outcomes) and `InquiryCount` (≈0.47, 0.36, 0.58) emerging as the strongest single drivers. In practice a modeler would now refit with `nfac=4`, monitor the residuals in the `scored` dataset, and promote the factor scores or coefficients into a production risk-decisioning pipeline.